# Reacting-flow fundamentals

The building blocks behind the combustor examples:

1. **`thermolib` HP equilibrium** -- the adiabatic flame temperature vs.
   equivalence ratio, straight from the NASA `thermo.inp` data (no network);
2. the perfect-gas **heat-release flame** -- a prescribed power `Qdot` raising the
   total temperature;
3. the reacting **equilibrium flame** -- frozen unburnt reactants in, equilibrium
   products out, the flame temperature emerging from the solve.


In [ ]:
import os, sys
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import matplotlib.pyplot as plt

from thermolib import ThermoInp, Thermo
from fns.composition import resolve_composition, enthalpy_mass
from fns.elements import catalog as cat
from fns.solver import solve
from fns.solver.control import states_table
from fns.thermo.api import EQ_FROZEN, EQ_KERNEL
from fns.thermo.configure import equilibrium, perfect_gas
from fns.derive import ES_MDOT, ES_P, ES_HT, ES_RHO, ES_U, ES_T, ES_M

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)

## 1. Adiabatic flame temperature vs. equivalence ratio (thermolib only)
Standalone chemical equilibrium: build a small H2/air library, then solve HP
equilibrium of the reactant elements at the reactant enthalpy.

In [ ]:
lib = ThermoInp(THERMO_INP).library(["H2", "O2", "N2", "H2O", "OH", "H", "O", "HO2", "NO"])
gas = Thermo(lib)

def h2_air(phi):
    """Stoichiometric-scaled H2/air mixture at equivalence ratio phi (mole)."""
    return {"H2": phi * 1.0, "O2": 0.5, "N2": 0.5 * 3.76}

phis = np.linspace(0.4, 1.6, 13)
Tad = []
for phi in phis:
    Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
    h = enthalpy_mass(lib, Y, 300.0)
    eq = gas.equilibrate_HP(Z, h, 101325.0, T_guess=2000.0)
    Tad.append(eq.T)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phis, Tad, "o-", color="#c0392b")
ax.set_xlabel("equivalence ratio  phi"); ax.set_ylabel("adiabatic flame T [K]")
ax.set_title("H2/air adiabatic flame temperature (HP equilibrium)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"peak Tad = {max(Tad):.1f} K near phi = {phis[int(np.argmax(Tad))]:.2f}")

## 2. Perfect-gas heat-release flame
A 2-port flame that adds a prescribed power `Qdot` [W]; the total temperature
rises by `Qdot / (mdot * cp)`, mass and the constant-area momentum flux are
conserved (a Rayleigh static-pressure drop appears).

In [ ]:
mdot, Tt, Qdot, A = 10.0, 300.0, 2.0e6, 0.05
cfg = perfect_gas(R_AIR, GAMMA)
els = [cat.mass_flow_inlet(mdot, Tt), cat.heat_release_flame(Qdot), cat.pressure_outlet(1.0e5)]
prob = cat.build_problem(cfg, els, [(0, 1, A), (1, 2, A)], mdot_ref=mdot, p_ref=1.0e5, h_ref=CP * Tt)
res = solve(prob); est = states_table(prob, res.x)
print("converged:", res.converged)
print(f"  Tt in/out : {est[ES_HT,0]/CP:.1f} -> {est[ES_HT,1]/CP:.1f} K  (predicted +{Qdot/(mdot*CP):.1f} K)")
print(f"  static T  : {est[ES_T,0]:.1f} -> {est[ES_T,1]:.1f} K")
print(f"  static p  : {est[ES_P,0]/1e3:.2f} -> {est[ES_P,1]/1e3:.2f} kPa (Rayleigh drop)")
imp = [est[ES_RHO,e]*est[ES_U,e]**2 + est[ES_P,e] for e in (0,1)]
print(f"  momentum flux rho*u^2 + p conserved: {imp[0]:.1f} vs {imp[1]:.1f} Pa")

## 3. Reacting equilibrium flame (single premixed stream)
Frozen H2/air approach -> equilibrium products.  'Ignition' is the per-edge
closure switch: the approach edge is `EQ_FROZEN`, the product edge `EQ_KERNEL`.
The premixed H2/air is a single **feed stream** (one transported mixture fraction,
discovered from the inlet composition at build time), and `solve(prob)` seeds
itself by propagating the feed through the network -- no hand-built guess.

In [ ]:
phi = 1.0
Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
h_react = enthalpy_mass(lib, Y, 300.0)
mdot, p, A = 1.0, 101325.0, 0.05
cfg = equilibrium(lib)
els = [
    cat.mass_flow_inlet(mdot, 300.0, composition=h2_air(phi), name="fuel-air"),
    cat.equilibrium_flame(name="flame"),
    cat.pressure_outlet(p, Tt_backflow=300.0, composition=h2_air(phi), name="out"),
]
prob = cat.build_problem(cfg, els, [(0, 1, A), (1, 2, A)], mdot_ref=mdot, p_ref=p,
                         h_ref=max(abs(h_react), 1e4), edge_models=[EQ_FROZEN, EQ_KERNEL])
res = solve(prob); est = states_table(prob, res.x)
ref = gas.equilibrate_HP(Z, h_react, est[ES_P, 1], T_guess=2000.0)
print("converged:", res.converged)
print(f"  unburnt T : {est[ES_T,0]:.1f} K     burnt T : {est[ES_T,1]:.1f} K")
print(f"  standalone HP-equilibrium reference : {ref.T:.1f} K")
print(f"  density dilatation rho_u/rho_b = {est[ES_RHO,0]/est[ES_RHO,1]:.2f}")

## Equivalence-ratio sweep through the network
The network flame temperature tracks the standalone adiabatic flame temperature.

In [ ]:
phis2 = np.linspace(0.5, 1.4, 10)
Tnet, Tref = [], []
for phi in phis2:
    Y, Z = resolve_composition(lib, h2_air(phi), basis="mole")
    h_react = enthalpy_mass(lib, Y, 300.0)
    els = [cat.mass_flow_inlet(1.0, 300.0, composition=h2_air(phi)),
           cat.equilibrium_flame(),
           cat.pressure_outlet(101325.0, Tt_backflow=300.0, composition=h2_air(phi))]
    prob = cat.build_problem(cfg, els, [(0,1,0.05),(1,2,0.05)], mdot_ref=1.0, p_ref=101325.0,
                             h_ref=max(abs(h_react),1e4), edge_models=[EQ_FROZEN, EQ_KERNEL])
    r = solve(prob)
    e = states_table(prob, r.x)
    Tnet.append(e[ES_T, 1])
    Tref.append(gas.equilibrate_HP(Z, h_react, e[ES_P, 1], T_guess=2000.0).T)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phis2, Tref, "-", lw=3, alpha=0.4, label="standalone HP equilibrium")
ax.plot(phis2, Tnet, "o", color="#c0392b", label="network flame")
ax.set_xlabel("equivalence ratio  phi"); ax.set_ylabel("burnt T [K]")
ax.set_title("Network flame vs. standalone equilibrium"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()